# TTPD Take-Home Assignment
### Tiny Town Police Department — Speeding Ticket Analysis

**Architecture:** Bronze → Silver → Gold Lakehouse

| Layer | Purpose |
|---|---|
| Bronze | Raw ingestion from CSV / XML / JSON into Delta tables |
| Silver | Cleaning, type casting, null handling, derived columns |
| Gold | Business aggregations that directly answer the 3 questions |

**Run cells from top to bottom, in order.**

## Cell 1 — Install Dependencies


In [1]:
# Colab now ships PySpark 4.0. We need delta-spark 4.0.0 to match.
!pip install delta-spark==4.0.0 --quiet


## Cell 2 — Upload and Unzip the Dataset

Upload `ttpd_data.zip` using the Colab Files panel on the left, then run this cell to unzip it.

In [3]:
import zipfile, os

# Unzip the dataset
with zipfile.ZipFile('ttpd_data.zip', 'r') as z:
    z.extractall('.')

print('Files extracted:')
files = os.listdir('ttpd_data')
print(f'  People CSVs:           {len([f for f in files if "people" in f])}')
print(f'  Automobile XMLs:       {len([f for f in files if "automobile" in f])}')
print(f'  Speeding Ticket JSONs: {len([f for f in files if "speeding" in f])}')

Files extracted:
  People CSVs:           52
  Automobile XMLs:       83
  Speeding Ticket JSONs: 36


## Cell 3 — Imports and Configuration

All imports and path constants in one place so they are easy to change if needed.

In [4]:
import os
import glob
from xml.etree import ElementTree as ET

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = "./ttpd_data"
DELTA_BASE = "./delta_tables"

BRONZE_PEOPLE_PATH   = f"{DELTA_BASE}/bronze/people"
BRONZE_AUTOS_PATH    = f"{DELTA_BASE}/bronze/automobiles"
BRONZE_TICKETS_PATH  = f"{DELTA_BASE}/bronze/speeding_tickets"

SILVER_PEOPLE_PATH   = f"{DELTA_BASE}/silver/people"
SILVER_AUTOS_PATH    = f"{DELTA_BASE}/silver/automobiles"
SILVER_TICKETS_PATH  = f"{DELTA_BASE}/silver/speeding_tickets"

GOLD_OFFICER_PATH    = f"{DELTA_BASE}/gold/officer_ticket_counts"
GOLD_MONTHLY_PATH    = f"{DELTA_BASE}/gold/monthly_ticket_counts"
GOLD_SPENDERS_PATH   = f"{DELTA_BASE}/gold/top_spenders"

print('Configuration loaded.')

Configuration loaded.


## Cell 4 — Create Spark Session

`configure_spark_with_delta_pip` automatically downloads the Delta Lake JAR files that Spark needs at runtime. This is the recommended way to set up Delta on local/Colab environments.

In [5]:
# configure_spark_with_delta_pip automatically downloads the Delta JAR that Spark needs.
# The two config keys tell Spark to use Delta's SQL extensions and table catalog.
# spark.sql.shuffle.partitions=4 keeps things fast on Colab's small dataset.

builder = (
    SparkSession
    .builder
    .appName("ttpd_takehome")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")  # hide noisy Spark logs

print('Spark session ready. PySpark version:', spark.version)


Spark session ready. PySpark version: 4.0.2


---
# BRONZE LAYER — Raw Ingestion

Load source files exactly as-is into Delta tables. No transformations here.

In [6]:
# ── Bronze: People (CSV) ───────────────────────────────────────────────────
# Files use pipe | as separator and double-quote wrapping.
# We pass a list of all matching files — Spark merges them into one DataFrame.

people_files = glob.glob(os.path.join(DATA_DIR, "*_people_*.csv"))

df_people_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .option("quote", '"')
    .option("inferSchema", "true")
    .csv(people_files)
)

df_people_bronze.write.format("delta").mode("overwrite").save(BRONZE_PEOPLE_PATH)

print(f"Bronze people: {df_people_bronze.count()} records")
df_people_bronze.printSchema()
df_people_bronze.show(3, truncate=False)

Bronze people: 7123 records
root
 |-- id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- address: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- profession: string (nullable = true)
 |-- company: string (nullable = true)
 |-- date_of_birth: date (nullable = true)

+------------------------------------+----------+---------+---+----------------------------+-----------------+--------------------+----------------------------+-------------+
|id                                  |first_name|last_name|sex|address                     |phone_number     |profession          |company                     |date_of_birth|
+------------------------------------+----------+---------+---+----------------------------+-----------------+--------------------+----------------------------+-------------+
|b1159589-c84c-446f-8db3-57372a64fbe1|Kyle      |Snyder   |M  |432 Thornton Land S

In [7]:
# ── Bronze: Automobiles (XML) ──────────────────────────────────────────────
# Spark doesn't natively read XML without a third-party package, so we parse
# using Python's ElementTree and convert to a Spark DataFrame.

auto_files = glob.glob(os.path.join(DATA_DIR, "*_automobiles_*.xml"))

records = []
for filepath in auto_files:
    tree = ET.parse(filepath)
    for auto in tree.getroot().findall("automobile"):
        records.append({
            "person_id":     auto.findtext("person_id"),
            "license_plate": auto.findtext("license_plate"),
            "vin":           auto.findtext("vin"),
            "color":         auto.findtext("color"),
            "year":          auto.findtext("year"),
        })

df_autos_bronze = spark.createDataFrame(records)
df_autos_bronze.write.format("delta").mode("overwrite").save(BRONZE_AUTOS_PATH)

print(f"Bronze automobiles: {df_autos_bronze.count()} records")
df_autos_bronze.printSchema()
df_autos_bronze.show(3, truncate=False)

Bronze automobiles: 10299 records
root
 |-- color: string (nullable = true)
 |-- license_plate: string (nullable = true)
 |-- person_id: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- year: string (nullable = true)

+----------+-------------+------------------------------------+-----------------+----+
|color     |license_plate|person_id                           |vin              |year|
+----------+-------------+------------------------------------+-----------------+----+
|WhiteSmoke|LLG7073      |f92b1fd2-66fb-42f2-8838-b6311f51fc51|GH74UK2U3LANMG7W3|1984|
|Peru      |2BVL 91      |d47edeb4-9754-4ee9-955b-783c66ad7a58|PJYK3T3J7X6YJ5DZK|2005|
|SteelBlue |FGD-451      |d47edeb4-9754-4ee9-955b-783c66ad7a58|1TWCEK053PPHURDXC|1998|
+----------+-------------+------------------------------------+-----------------+----+
only showing top 3 rows


In [8]:
# ── Bronze: Speeding Tickets (JSON) ───────────────────────────────────────
# Each file: { "speeding_tickets": [ {ticket}, {ticket}, ... ] }
# explode() turns the array column into individual rows.

ticket_files = glob.glob(os.path.join(DATA_DIR, "*_speeding_tickets_*.json"))

df_raw = (
    spark.read
    .option("multiLine", "true")
    .json(ticket_files)
)

df_tickets_bronze = (
    df_raw
    .select(F.explode("speeding_tickets").alias("ticket"))
    .select("ticket.*")   # flatten nested struct to individual columns
)

df_tickets_bronze.write.format("delta").mode("overwrite").save(BRONZE_TICKETS_PATH)

print(f"Bronze tickets: {df_tickets_bronze.count()} records")
df_tickets_bronze.printSchema()
df_tickets_bronze.show(3, truncate=False)

Bronze tickets: 16123 records
root
 |-- id: string (nullable = true)
 |-- license_plate: string (nullable = true)
 |-- officer_id: string (nullable = true)
 |-- recorded_mph_over_limit: long (nullable = true)
 |-- school_zone_ind: boolean (nullable = true)
 |-- speed_limit: long (nullable = true)
 |-- ticket_time: string (nullable = true)
 |-- work_zone_ind: boolean (nullable = true)

+------------------------------------+-------------+------------------------------------+-----------------------+---------------+-----------+-------------------+-------------+
|id                                  |license_plate|officer_id                          |recorded_mph_over_limit|school_zone_ind|speed_limit|ticket_time        |work_zone_ind|
+------------------------------------+-------------+------------------------------------+-----------------------+---------------+-----------+-------------------+-------------+
|3339a54c-1147-4eda-8720-1a5fad920a50|23EU645      |0e1f5d48-2bb1-45fb-9b46-01a907d6

---
# SILVER LAYER — Cleaning & Preparation

Read from Bronze. Fix types, handle nulls, add helper columns. No business logic yet.

In [9]:
# ── Silver: People ─────────────────────────────────────────────────────────

df_people_silver = (
    spark.read.format("delta").load(BRONZE_PEOPLE_PATH)

    # Drop rows with no ID — they can't be joined to anything
    .filter(F.col("id").isNotNull())

    # UPPER-case profession so 'Police officer' == 'Police Officer'
    .withColumn("profession", F.upper(F.trim(F.col("profession"))))

    # Parse date string → DateType
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))

    # Clean name fields
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name",  F.trim(F.col("last_name")))

    # Full name for display
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))

    # Boolean flag: True if this person is a police officer
    .withColumn("is_officer", F.col("profession") == "POLICE OFFICER")
)

df_people_silver.write.format("delta").mode("overwrite").save(SILVER_PEOPLE_PATH)

officer_count = df_people_silver.filter(F.col("is_officer")).count()
print(f"Silver people: {df_people_silver.count()} records | Officers found: {officer_count}")
df_people_silver.filter(F.col("is_officer")).show(3, truncate=False)

Silver people: 7123 records | Officers found: 55
+------------------------------------+----------+---------+---+--------------------------+-------------+--------------+-------+-------------+--------------+----------+
|id                                  |first_name|last_name|sex|address                   |phone_number |profession    |company|date_of_birth|full_name     |is_officer|
+------------------------------------+----------+---------+---+--------------------------+-------------+--------------+-------+-------------+--------------+----------+
|51226066-9f3d-482f-a245-9a9b7c1698c8|Tyler     |Buchanan |M  |136 Daniel Plaza          |851.352.4381 |POLICE OFFICER|TTPD   |1921-05-10   |Tyler Buchanan|true      |
|368eb483-c619-4c08-9421-4168d1b78625|Heather   |Rhodes   |F  |99 Ashley Landing         |(683)835-0442|POLICE OFFICER|TTPD   |1978-03-22   |Heather Rhodes|true      |
|2cd8fc26-c03b-428c-ab87-bdecb40e9507|John      |Mcdaniel |M  |566 Murphy Creek Suite 712|975.704.1589 |POLICE 

In [10]:
# ── Silver: Automobiles ────────────────────────────────────────────────────

df_autos_silver = (
    spark.read.format("delta").load(BRONZE_AUTOS_PATH)
    .filter(F.col("license_plate").isNotNull() & F.col("person_id").isNotNull())
    .withColumn("year", F.col("year").cast(IntegerType()))
    .withColumn("license_plate", F.trim(F.col("license_plate")))
)

df_autos_silver.write.format("delta").mode("overwrite").save(SILVER_AUTOS_PATH)
print(f"Silver automobiles: {df_autos_silver.count()} records")
df_autos_silver.show(3, truncate=False)

Silver automobiles: 10299 records
+-----------+-------------+------------------------------------+-----------------+----+
|color      |license_plate|person_id                           |vin              |year|
+-----------+-------------+------------------------------------+-----------------+----+
|BurlyWood  |7E 04216     |2d4e0b42-6e61-4195-b0b8-d5394e249171|5JGHJZCU4LMZZ6L4U|1970|
|DarkMagenta|5533 ZD      |e1b3543f-d242-49c2-9ec1-b5a16dbd0c62|ZBH3CR6E4SPHE9R0U|1992|
|SeaShell   |4GC 166      |e1b3543f-d242-49c2-9ec1-b5a16dbd0c62|2N06HBHH8BCR0HVPX|1993|
+-----------+-------------+------------------------------------+-----------------+----+
only showing top 3 rows


In [11]:
# ── Silver: Speeding Tickets ───────────────────────────────────────────────

df_tickets_silver = (
    spark.read.format("delta").load(BRONZE_TICKETS_PATH)
    .filter(F.col("license_plate").isNotNull() & F.col("officer_id").isNotNull())

    # Parse timestamp string into TimestampType
    .withColumn("ticket_time", F.to_timestamp(F.col("ticket_time"), "yyyy-MM-dd HH:mm:ss"))

    # Derive year and YYYY-MM string for aggregations
    .withColumn("ticket_year",      F.year(F.col("ticket_time")))
    .withColumn("ticket_month_str", F.date_format(F.col("ticket_time"), "yyyy-MM"))

    # Ensure correct boolean types
    .withColumn("school_zone_ind", F.col("school_zone_ind").cast(BooleanType()))
    .withColumn("work_zone_ind",   F.col("work_zone_ind").cast(BooleanType()))

    # Apply fee schedule:
    #   school=T AND work=T  → $120
    #   school=T OR  work=T  → $60
    #   neither              → $30
    .withColumn(
        "ticket_fee",
        F.when(F.col("school_zone_ind") & F.col("work_zone_ind"),  120)
         .when(F.col("school_zone_ind") | F.col("work_zone_ind"),   60)
         .otherwise(30)
    )
)

df_tickets_silver.write.format("delta").mode("overwrite").save(SILVER_TICKETS_PATH)
print(f"Silver tickets: {df_tickets_silver.count()} records")
df_tickets_silver.show(3, truncate=False)

Silver tickets: 16123 records
+------------------------------------+-------------+------------------------------------+-----------------------+---------------+-----------+-------------------+-------------+-----------+----------------+----------+
|id                                  |license_plate|officer_id                          |recorded_mph_over_limit|school_zone_ind|speed_limit|ticket_time        |work_zone_ind|ticket_year|ticket_month_str|ticket_fee|
+------------------------------------+-------------+------------------------------------+-----------------------+---------------+-----------+-------------------+-------------+-----------+----------------+----------+
|3339a54c-1147-4eda-8720-1a5fad920a50|23EU645      |0e1f5d48-2bb1-45fb-9b46-01a907d6f546|18                     |false          |55         |2021-08-31 07:59:38|true         |2021       |2021-08         |60        |
|e753a5b1-f9ac-4284-94da-d9f4cf25acf3|IQD4517      |e2155846-cecb-4713-9da8-8cface393ddf|26               

---
# GOLD LAYER — Analytics Aggregations

Read from Silver. Join, aggregate, answer the questions.

In [12]:
# ── Gold: Q1 — Officer Ticket Counts ──────────────────────────────────────
# Join tickets → people (officers only), count per officer

df_tickets_s  = spark.read.format("delta").load(SILVER_TICKETS_PATH)
df_people_s   = spark.read.format("delta").load(SILVER_PEOPLE_PATH)

df_officer_gold = (
    df_tickets_s
    .join(
        df_people_s.filter(F.col("is_officer")),
        df_tickets_s["officer_id"] == df_people_s["id"],
        how="inner"
    )
    .groupBy("officer_id", "full_name")
    .agg(F.count("*").alias("tickets_issued"))
    .orderBy(F.col("tickets_issued").desc())
)

df_officer_gold.write.format("delta").mode("overwrite").save(GOLD_OFFICER_PATH)
print(f"Gold officer table: {df_officer_gold.count()} officers")

Gold officer table: 42 officers


In [13]:
# ── Gold: Q2 — Monthly Ticket Counts ──────────────────────────────────────

df_monthly_gold = (
    spark.read.format("delta").load(SILVER_TICKETS_PATH)
    .groupBy("ticket_year", "ticket_month_str")
    .agg(F.count("*").alias("ticket_count"))
    .orderBy(F.col("ticket_count").desc())
)

df_monthly_gold.write.format("delta").mode("overwrite").save(GOLD_MONTHLY_PATH)
print(f"Gold monthly table: {df_monthly_gold.count()} months")

Gold monthly table: 48 months


In [14]:
# ── Gold: Q3 — Top 10 Spenders ────────────────────────────────────────────
# ticket → automobile (via license_plate) → people (via person_id)

df_autos_s = spark.read.format("delta").load(SILVER_AUTOS_PATH)

df_spenders_gold = (
    df_tickets_s
    .join(df_autos_s,  on="license_plate", how="inner")
    .join(df_people_s, df_autos_s["person_id"] == df_people_s["id"], how="inner")
    .groupBy("person_id", "full_name")
    .agg(F.sum("ticket_fee").alias("total_fees_paid"))
    .orderBy(F.col("total_fees_paid").desc())
    .limit(10)
)

df_spenders_gold.write.format("delta").mode("overwrite").save(GOLD_SPENDERS_PATH)
print(f"Gold spenders table: {df_spenders_gold.count()} people")

Gold spenders table: 10 people


---
# FINAL ANSWERS

In [15]:
print("=" * 60)
print("QUESTION 1: Which police officer issued the most tickets?")
print("=" * 60)
print("Officers are identified by profession = 'POLICE OFFICER' in the people table.")
print()

df_officers = spark.read.format("delta").load(GOLD_OFFICER_PATH)
top = df_officers.limit(1).collect()[0]
print(f"  Answer: {top['full_name']}  —  {top['tickets_issued']} tickets issued")
print()
print("Top 5 officers:")
df_officers.show(5, truncate=False)

QUESTION 1: Which police officer issued the most tickets?
Officers are identified by profession = 'POLICE OFFICER' in the people table.

  Answer: Barbara Cervantes  —  419 tickets issued

Top 5 officers:
+------------------------------------+-----------------+--------------+
|officer_id                          |full_name        |tickets_issued|
+------------------------------------+-----------------+--------------+
|f6c8ecff-14c4-410f-ac7a-adb9214ed6b8|Barbara Cervantes|419           |
|368eb483-c619-4c08-9421-4168d1b78625|Heather Rhodes   |417           |
|274e40c3-fa93-4f44-8a4c-0e6c36917cea|Kimberly Smith   |415           |
|3e99107e-ed1c-4895-b78a-85d270e0559a|Diane Fisher     |415           |
|3752c341-8470-455a-889d-c0fc2d1f5283|Louis Drake      |405           |
+------------------------------------+-----------------+--------------+
only showing top 5 rows


In [16]:
print("=" * 60)
print("QUESTION 2: Top 3 months with most speeding tickets?")
print("=" * 60)

df_monthly = spark.read.format("delta").load(GOLD_MONTHLY_PATH)
top3 = df_monthly.limit(3).collect()
for i, r in enumerate(top3, 1):
    print(f"  #{i}: {r['ticket_month_str']}  →  {r['ticket_count']} tickets")

print()
print("BONUS — Year-by-year totals:")
df_monthly.groupBy("ticket_year").agg(
    F.sum("ticket_count").alias("yearly_total")
).orderBy("ticket_year").show()

print("BONUS — All months ranked:")
df_monthly.show(50, truncate=False)

QUESTION 2: Top 3 months with most speeding tickets?
  #1: 2023-12  →  1258 tickets
  #2: 2022-12  →  824 tickets
  #3: 2021-12  →  803 tickets

BONUS — Year-by-year totals:
+-----------+------------+
|ticket_year|yearly_total|
+-----------+------------+
|       2020|        1601|
|       2021|        4023|
|       2022|        4077|
|       2023|        6422|
+-----------+------------+

BONUS — All months ranked:
+-----------+----------------+------------+
|ticket_year|ticket_month_str|ticket_count|
+-----------+----------------+------------+
|2023       |2023-12         |1258        |
|2022       |2022-12         |824         |
|2021       |2021-12         |803         |
|2023       |2023-08         |774         |
|2023       |2023-01         |647         |
|2023       |2023-07         |641         |
|2023       |2023-09         |631         |
|2023       |2023-06         |507         |
|2022       |2022-08         |493         |
|2021       |2021-08         |477         |
|2022     

In [17]:
print("=" * 60)
print("QUESTION 3: Top 10 biggest spenders on speeding tickets")
print("=" * 60)
print("Fee schedule: base=$30 | +school zone=$60 | +work zone=$60 | both=$120")
print()

df_spenders = spark.read.format("delta").load(GOLD_SPENDERS_PATH)
for i, r in enumerate(df_spenders.collect(), 1):
    print(f"  #{i:2d}  {r['full_name']:<25}  ${r['total_fees_paid']:.0f}")

print()
df_spenders.show(10, truncate=False)

QUESTION 3: Top 10 biggest spenders on speeding tickets
Fee schedule: base=$30 | +school zone=$60 | +work zone=$60 | both=$120

  # 1  Charles Dunn               $660
  # 2  Kimberly Mcdowell          $660
  # 3  Ariel Smith                $630
  # 4  Pamela Young               $630
  # 5  Avery Smith                $600
  # 6  Tracy Casey                $600
  # 7  Carla Robinson             $570
  # 8  Adrian May                 $570
  # 9  Elizabeth Zuniga           $540
  #10  Jerry Little               $540

+------------------------------------+-----------------+---------------+
|person_id                           |full_name        |total_fees_paid|
+------------------------------------+-----------------+---------------+
|9c7f69d1-5d9b-4f62-85da-93b5f21bc359|Charles Dunn     |660            |
|cc2ea84b-c615-47bc-a7ab-9fb1a012bc53|Kimberly Mcdowell|660            |
|12fd23fd-7b1b-4d96-813e-0f1b1e7faec0|Ariel Smith      |630            |
|56ec719d-0aa4-4968-97b6-9b2c48777600|Pamel

In [18]:
spark.stop()
print("Pipeline complete. Spark session stopped.")

Pipeline complete. Spark session stopped.
